# Grad-CAM — Mapas de Calor para Interpretáncia dos Modelos

Este notebook gera **Grad-CAM** (Gradient-weighted Class Activation Mapping) para visualizar quais regiões das folhas de soja os modelos CNN estão focando ao tomar decisões de classificação.

Suporta os 4 modelos: ResNet-101, DenseNet-201, VGG-19 e MobileNetV3-Large.

In [ ]:
import os, torch, numpy as np, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

# ============================================
# CONFIGURAÇÃO
# ============================================
MODEL_PATH = os.getenv("TRAINED_MODEL_PATH", "./models/best_model.pt")
IMAGE_DIR = os.getenv("DATASET_PATH", "./data/DADOS-DIVIDIDOS/test/diseases")
MODEL_NAME = "densenet201"  # Altere para: resnet101, vgg19, mobilenetv3

# Hiperparâmetros ótimos de cada modelo
HYPERPARAMS = {
    "resnet101": {"dropout1": 0.4720, "dropout2": 0.2132, "fc1": 256, "fc2": 256},
    "densenet201": {"dropout1": 0.3373, "dropout2": 0.2673, "fc1": 512, "fc2": 256},
    "vgg19": {"dropout1": 0.4751, "dropout2": 0.2508, "fc1": 1024, "fc2": 256},
    "mobilenetv3": {"dropout1": 0.4730, "dropout2": 0.3255, "fc1": 512, "fc2": 512},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Carregamento do Modelo

In [ ]:
def load_model(model_name, model_path, device):
    hp = HYPERPARAMS[model_name]
    if model_name == "resnet101":
        model = models.resnet101(pretrained=False)
        nf = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(hp['dropout1']), nn.Linear(nf, hp['fc1']), nn.ReLU(),
            nn.Dropout(hp['dropout2']), nn.Linear(hp['fc1'], hp['fc2']), nn.ReLU(),
            nn.Linear(hp['fc2'], 2))
    elif model_name == "densenet201":
        model = models.densenet201(pretrained=False)
        nf = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Dropout(hp['dropout1']), nn.Linear(nf, hp['fc1']), nn.ReLU(),
            nn.Dropout(hp['dropout2']), nn.Linear(hp['fc1'], hp['fc2']), nn.ReLU(),
            nn.Linear(hp['fc2'], 2))
    elif model_name == "vgg19":
        model = models.vgg19(pretrained=False)
        nf = model.classifier[0].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(hp['dropout1']), nn.Linear(nf, hp['fc1']), nn.ReLU(),
            nn.Dropout(hp['dropout2']), nn.Linear(hp['fc1'], hp['fc2']), nn.ReLU(),
            nn.Linear(hp['fc2'], 2))
    elif model_name == "mobilenetv3":
        model = models.mobilenet_v3_large(pretrained=False)
        nf = model.classifier[0].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(hp['dropout1']), nn.Linear(nf, hp['fc1']), nn.ReLU(),
            nn.Dropout(hp['dropout2']), nn.Linear(hp['fc1'], hp['fc2']), nn.ReLU(),
            nn.Linear(hp['fc2'], 2), nn.Softmax(dim=1))
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    return model.to(device).eval()

def get_target_layer(model, model_name):
    if model_name == "resnet101": return model.layer4[-1].conv3
    elif model_name == "densenet201": return model.features.denseblock4.denselayer32.conv2
    elif model_name == "vgg19": return model.features[-1]
    elif model_name == "mobilenetv3": return model.features[-1]

model = load_model(MODEL_NAME, MODEL_PATH, device)
print(f"Modelo {MODEL_NAME} carregado com sucesso!")

## 2. Implementação do Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, input, output):
        self.activations = output.detach()

    def _bwd_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1.0
        output.backward(gradient=one_hot)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.clamp(cam, min=0)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), target_class

target_layer = get_target_layer(model, MODEL_NAME)
grad_cam = GradCAM(model, target_layer)
print("Grad-CAM inicializado!")

## 3. Geração dos Mapas de Calor

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class_names = ['diseases', 'healthy']

# Selecionar imagens para análise
image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))][:8]

fig, axes = plt.subplots(len(image_files), 3, figsize=(15, 5 * len(image_files)))
if len(image_files) == 1:
    axes = axes.reshape(1, -1)

for idx, filename in enumerate(image_files):
    img_path = os.path.join(IMAGE_DIR, filename)
    image = Image.open(img_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    cam, pred_class = grad_cam.generate(input_tensor)
    cam_resized = np.array(Image.fromarray(cam).resize((224, 224), Image.BILINEAR))
    img_display = image.resize((224, 224))
    
    axes[idx, 0].imshow(img_display)
    axes[idx, 0].set_title(f'Original ({class_names[pred_class]})')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(cam_resized, cmap='jet')
    axes[idx, 1].set_title('Grad-CAM')
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(img_display)
    axes[idx, 2].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[idx, 2].set_title('Sobreposição')
    axes[idx, 2].axis('off')

plt.suptitle(f'Grad-CAM — {MODEL_NAME.upper()}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'grad_cam_{MODEL_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mapas de calor salvos em: grad_cam_{MODEL_NAME}.png")